In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import copy

In [ ]:
class MTP(nn.Module):
    def __init__(self, model, predict_tokens_num=5, mtp_lambda=1.0, freeze_base_model=True):
        super().__init__()
        self.predict_tokens_num = predict_tokens_num
        self.mtp_lambda = mtp_lambda
        
        # 主模型
        self.main_model = model.base_model
        if freeze_base_model:
            for param in self.main_model.parameters():
                param.requires_grad = False
        
        H = self.main_model.config.hidden_size
        V = self.main_model.config.vocab_size
        
        # 按照图像设计：每个MTP模块独立
        self.mtp_modules = nn.ModuleList()
        for i in range(1, predict_tokens_num):  # mtp1, mtp2, mtp3, mtp4
            self.mtp_modules.append(MTPModule(H, self.main_model.layers[0]))
        
        # 每个模块有自己的输出头（按照图像）
        self.output_heads = nn.ModuleList([
            MTPHead(H, V, self.main_model.get_input_embeddings().weight) 
            for _ in range(predict_tokens_num)
        ])
        
        # 添加RMSNorm层（图像中有）
        self.rms_norm = nn.LayerNorm(H, eps=1e-6)  # 近似RMSNorm

    def forward(self, input_ids, attention_mask=None, training=True):
        outputs = {}
        
        # 主模型前向
        with torch.no_grad():
            main_outputs = self.main_model(input_ids=input_ids, attention_mask=attention_mask)
            main_hidden = main_outputs.last_hidden_state
        
        # 主输出（t+1预测）
        main_hidden_norm = self.rms_norm(main_hidden)
        main_logits = self.output_heads[0](main_hidden_norm)
        outputs['head_main'] = main_logits
        
        if not training:
            return outputs
        
        # 每个MTP模块独立计算（按照图像设计）
        input_embed = self.main_model.get_input_embeddings()(input_ids)
        input_embed_norm = self.rms_norm(input_embed)
        
        for i in range(1, self.predict_tokens_num):
            # 每个MTP模块接收主模型的hidden state和输入embedding
            mtp_hidden = self.mtp_modules[i-1](
                main_hidden_norm,  # 主模型的归一化隐藏状态
                input_embed_norm,   # 输入token的归一化嵌入
                attention_mask=attention_mask
            )
            mtp_logits = self.output_heads[i](mtp_hidden)
            outputs[f'mtp_head_{i}'] = mtp_logits
        
        return outputs

    def compute_loss(self, outputs: dict, labels: torch.Tensor):
        device = labels.device
        total_loss = 0
        losses = {}
        
        # 主损失（t+1）
        main_logits = outputs['head_main']
        B, S, V = main_logits.shape
        
        main_logits_flat = main_logits[:, :-1, :].reshape(-1, V)
        main_targets = labels[:, 1:].reshape(-1)
        main_loss = F.cross_entropy(main_logits_flat, main_targets, ignore_index=-100)
        losses['main'] = main_loss
        total_loss += main_loss
        
        # MTP损失（t+2, t+3, ...）
        mtp_losses = []
        for i in range(1, self.predict_tokens_num):
            key = f'mtp_head_{i}'
            if key not in outputs:
                continue
                
            mtp_logits = outputs[key]
            offset = i + 1  # mtp_head_1 -> t+2, mtp_head_2 -> t+3, etc.
            valid_len = S - offset
            
            if valid_len <= 0:
                continue
                
            logits = mtp_logits[:, :valid_len, :].reshape(-1, V)
            targets = labels[:, offset:offset+valid_len].reshape(-1)
            loss_i = F.cross_entropy(logits, targets, ignore_index=-100)
            mtp_losses.append(loss_i)
            losses[f'mtp_{i}'] = loss_i
        
        if mtp_losses:
            mtp_loss = torch.stack(mtp_losses).mean()
            total_loss += self.mtp_lambda * mtp_loss
            losses['mtp_total'] = mtp_loss
        else:
            mtp_loss = torch.tensor(0.0, device=device)
            losses['mtp_total'] = mtp_loss
        
        losses['total'] = total_loss
        return losses

class MTPModule(nn.Module):
    """MTP模块"""
    def __init__(self, hidden_size, transformer_block):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, eps=1e-6)  # RMSNorm替代
        self.norm2 = nn.LayerNorm(hidden_size, eps=1e-6)
        self.linear_proj = nn.Linear(2 * hidden_size, hidden_size)
        self.transformer_block = copy.deepcopy(transformer_block)
    
    def forward(self, main_hidden, input_embed, attention_mask=None):
        # 归一化（图像中的RMSNorm）
        main_hidden_norm = self.norm1(main_hidden)
        input_embed_norm = self.norm2(input_embed)
        
        # 拼接和线性投影
        concatenated = torch.cat([main_hidden_norm, input_embed_norm], dim=-1)
        projected = self.linear_proj(concatenated)
        
        # 生成位置ID（Qwen2模型需要）
        batch_size, seq_length = projected.shape[:2]
        position_ids = torch.arange(seq_length, dtype=torch.long, device=projected.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)

        # Transformer块 - 传递所有必要参数
        if hasattr(self.transformer_block, 'forward'):
            try:
                # 尝试使用正确的方法调用transformer块
                output = self.transformer_block(
                    hidden_states=projected,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    use_cache=False  # 训练时不需要缓存
                )
                if isinstance(output, tuple):
                    output = output[0]
            except Exception as e:
                # 如果复杂调用失败，使用简单的前向传播
                print(f"警告: 使用简化transformer调用，错误: {e}")
                output = projected
        else:
            output = projected

        
        return output

class MTPHead(nn.Module):
    def __init__(self, hidden_size, vocab_size, tie_embedding=None):
        super().__init__()
        self.linear = nn.Linear(hidden_size, vocab_size)
        if tie_embedding is not None:
            self.linear.weight = tie_embedding
    
    def forward(self, hidden_states):
        return self.linear(hidden_states)

In [ ]:
model_path = "../model/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_path)
mtp_model = MTP(model, predict_tokens_num=5)
total_params = sum(p.numel() for p in mtp_model.parameters() if p.requires_grad)
print(f"可训练参数: {total_params / 1e6:.2f}M")

In [ ]:
mtp_model

In [ ]:

class MTPTrainer:
    def __init__(self, model, train_dataloader, optimizer, device='cuda', 
                 mtp_lambda=1.0, grad_clip=1.0, save_path='../model/mtp/checkpoint'):
        self.model = model
        self.train_dataloader = train_dataloader
        self.optimizer = optimizer
        self.device = device
        self.mtp_lambda = mtp_lambda
        self.grad_clip = grad_clip
        self.save_path = save_path
        
        # 创建保存目录
        os.makedirs(save_path, exist_ok=True)
        
        self.scaler = GradScaler()
        self.model.to(device)
        
    def train_epoch(self, epoch, writer, print_step=10, save_step=1000):
        """训练一个epoch"""
        self.model.train()
        total_steps = len(self.train_dataloader)
        
        for step, batch in enumerate(self.train_dataloader):
            self.optimizer.zero_grad()
            
            # 移动到设备
            input_ids = batch['input_ids'].to(self.device)
            labels = batch['labels'].to(self.device)
            attention_mask = batch.get('attention_mask', None)
            if attention_mask is not None:
                attention_mask = attention_mask.to(self.device)
            
            # 混合精度训练
            with autocast():
                outputs = self.model(input_ids, attention_mask=attention_mask, training=True)
                losses = self.model.compute_loss(outputs, labels)
                total_loss = losses['total']
            
            # 反向传播
            self.scaler.scale(total_loss).backward()
            self.scaler.unscale_(self.optimizer)
            
            # 梯度裁剪
            if self.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            
            # 优化器步进
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            # 记录损失
            current_step = epoch * total_steps + step
            if current_step % print_step == 0:
                self._log_losses(losses, current_step, writer, epoch, step)
            
            # 保存检查点
            if current_step % save_step == 0 and current_step > 0:
                self._save_checkpoint(current_step)
            
            # 清理GPU缓存
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    def _log_losses(self, losses, step, writer, epoch, batch_step):
        """记录损失到tensorboard和打印"""
        main_loss = losses['main'].item()
        mtp_total = losses['mtp_total'].item()
        total_loss = losses['total'].item()
        
        # TensorBoard记录
        writer.add_scalar('train/main_loss', main_loss, step)
        writer.add_scalar('train/mtp_total_loss', mtp_total, step)
        writer.add_scalar('train/total_loss', total_loss, step)
        
        # 记录各个MTP头的损失
        for key, loss in losses.items():
            if key.startswith('mtp_') and not key.endswith('_total'):
                writer.add_scalar(f'train/{key}', loss.item(), step)
        
        # 打印信息
        print(f"[Epoch {epoch+1} Batch {batch_step}] Step {step}, "
              f"main_loss={main_loss:.4f}, mtp_total={mtp_total:.4f}, total_loss={total_loss:.4f}")
    
    def _save_checkpoint(self, step):
        """保存检查点"""
        checkpoint_path = f"{self.save_path}/checkpoint_{step}.pt"
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scaler_state_dict': self.scaler.state_dict(),
            'step': step,
            'mtp_lambda': self.mtp_lambda
        }, checkpoint_path)
        print(f"检查点已保存: {checkpoint_path}")
    
    def train(self, epochs, writer, print_step=10, save_step=1000):
        """完整训练流程"""
        for epoch in range(epochs):
            print(f"开始训练第 {epoch+1}/{epochs} 轮")
            self.train_epoch(epoch, writer, print_step, save_step)
        
        # 训练完成保存最终模型
        final_path = f"{self.save_path}/final_model.pt"
        torch.save(self.model.state_dict(), final_path)
        print(f"最终模型已保存: {final_path}")

# 数据相关
class ImprovedMTPDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=512):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, index):
        sample = self.dataset[index]
        user = sample["input"]
        assistant = sample["target"]
        
        # 构造对话格式
        messages = [
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant}
        ]
        
        # 使用tokenizer的apply_chat_template
        text = self.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        
        # Tokenize
        encoded = self.tokenizer(
            text, 
            max_length=self.max_length, 
            truncation=True, 
            padding=False,
            return_tensors=None
        )
        
        # 创建labels（仅对assistant部分计算loss）
        input_ids = encoded['input_ids']
        labels = input_ids.copy()
        
        # 找到assistant开始的位置
        assistant_start = text.find(assistant)
        if assistant_start != -1:
            # 标记user部分为-100
            prefix_text = text[:assistant_start]
            prefix_tokens = self.tokenizer(prefix_text, add_special_tokens=False)['input_ids']
            for i in range(len(prefix_tokens)):
                if i < len(labels):
                    labels[i] = -100
        
        return {
            "input_ids": input_ids,
            "labels": labels,
            "attention_mask": encoded.get('attention_mask', [1] * len(input_ids))
        }

class ImprovedDataCollator:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of
    
    def __call__(self, features):
        max_len = max(len(f['input_ids']) for f in features)
        
        # 填充到8的倍数（优化GPU效率）
        if self.pad_to_multiple_of > 0:
            max_len = ((max_len + self.pad_to_multiple_of - 1) // self.pad_to_multiple_of) * self.pad_to_multiple_of
        
        batch = {}
        for key in ['input_ids', 'labels', 'attention_mask']:
            batch[key] = []
            for feature in features:
                padded = feature[key] + [self._get_pad_value(key)] * (max_len - len(feature[key]))
                batch[key].append(padded[:max_len])  # 确保不超过max_len
        
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}
    
    def _get_pad_value(self, key):
        if key == 'labels':
            return -100
        elif key == 'attention_mask':
            return 0
        else:  # input_ids
            return self.tokenizer.pad_token_id

In [ ]:
# 1. 加载模型和tokenizer
model_path = "../model/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

# 2. 创建改进的MTP模型
mtp_model = MTP(
    model=model, 
    predict_tokens_num=5,  # 预测t+1到t+4
    mtp_lambda=1.0,
    freeze_base_model=True
)

# 3. 准备数据
ds = load_dataset("YeungNLP/firefly-train-1.1M")
dataset = ds["train"].shuffle(42).select(range(2000))  # 小批量测试

train_dataset = ImprovedMTPDataset(dataset, tokenizer, max_length=512)
collator = ImprovedDataCollator(tokenizer)
train_loader = DataLoader(train_dataset, batch_size=2, collate_fn=collator, shuffle=True)

# 4. 设置优化器
optimizer = torch.optim.AdamW(
    mtp_model.parameters(), 
    lr=1e-4,
    weight_decay=0.01
)

# 5. 创建trainer和开始训练
writer = SummaryWriter('../model/mtp/runs')
trainer = MTPTrainer(
    model=mtp_model,
    train_dataloader=train_loader,
    optimizer=optimizer,
    device='cuda',
    mtp_lambda=1.0,
    save_path='../model/mtp/checkpoint'
)

# 6. 开始训练
trainer.train(epochs=3, writer=writer, print_step=10, save_step=100)

writer.close()